In [11]:
# import
import pandas as pd
import requests
import time
import json
import os
import re

In [ ]:
# API configuration
API_DELAY = 1  # seconds between API calls to avoid rate limiting


def get_iocsANDexploits(cve_id):
    """Enhanced IOC extraction from multiple sources with better OTX parsing"""
    time.sleep(API_DELAY)
    try:
        # First try AlienVault OTX
        otx_url = f"https://otx.alienvault.com/api/v1/indicators/cve/{cve_id}"
        otx_response = requests.get(otx_url)
        otx_response.raise_for_status()
        otx_data = otx_response.json()
        
        iocs = {
            'IPs': set(),
            'Domains': set(),
            'URLs': set(),
            'Hashes': set(),
            'FileNames': set(),
            'Emails': set(),
            'Pulse_count': 0,
            'Pulse_samples': []
        }

        # Parse OTX pulses more thoroughly
        if 'pulse_info' in otx_data and 'pulses' in otx_data['pulse_info']:
            pulses = otx_data['pulse_info']['pulses']
            iocs['Pulse_count'] = len(pulses)
            
            # Sample first 3 pulse names for context
            iocs['Pulse_samples'] = [pulse.get('name', '') for pulse in pulses[:3]]
            
            # Try to get indicators from pulse details
            for pulse in pulses[:5]:  # Limit to first 5 pulses to avoid timeout
                try:
                    pulse_id = pulse['id']
                    pulse_detail_url = f"https://otx.alienvault.com/api/v1/pulses/{pulse_id}"
                    pulse_response = requests.get(pulse_detail_url, timeout=10)
                    pulse_data = pulse_response.json()
                    
                    if 'indicators' in pulse_data:
                        for indicator in pulse_data['indicators']:
                            indicator_type = indicator.get('type', '')
                            indicator_value = indicator.get('indicator', '')
                            
                            if indicator_type == 'IPv4':
                                iocs['IPs'].add(indicator_value)
                            elif indicator_type == 'domain':
                                iocs['Domains'].add(indicator_value)
                            elif indicator_type == 'URL':
                                iocs['URLs'].add(indicator_value)
                            elif indicator_type in ['FileHash-MD5', 'FileHash-SHA1', 'FileHash-SHA256']:
                                iocs['Hashes'].add(indicator_value)
                            elif indicator_type == 'filename':
                                iocs['FileNames'].add(indicator_value)
                            elif indicator_type == 'email':
                                iocs['Emails'].add(indicator_value)
                except Exception as pulse_error:
                    print(f"Error processing pulse {pulse_id}: {str(pulse_error)}")
                    continue

        # If still no IOCs, try additional sources
        if not any(iocs.values()):  # If all IOC lists are empty
            print(f"No IOCs found in OTX for {cve_id}, trying threatminer as additional sources")
            iocs.update(get_iocs_from_secondary_sources(cve_id))
        
        # Convert sets to lists
        for key in iocs:
            if isinstance(iocs[key], set):
                iocs[key] = list(iocs[key])
                
        return {
            'source': 'OTX alien vault / threatminer if alien vault empty',
            'Exploits': otx_response.json().get('exploits', []),
            'pulse_context': {
                'count': iocs.pop('Pulse_count'),
                'samples': iocs.pop('Pulse_samples')
            },
            'iocs': iocs
        }
        
    except Exception as e:
        print(f"Error fetching IOCs for {cve_id}: {str(e)}")
        return None

def get_iocs_from_secondary_sources(cve_id):
    """Fallback IOC extraction from other sources"""
    secondary_iocs = {
        'Static_Item_Source:': "ThreatMiner",
        'IPs': set(),
        'Domains': set(),
        'URLs': set(),
        'Hashes': set()
    }
    
    try:
        # Try ThreatMiner
        tm_url = f"https://api.threatminer.org/v2/vulnerability.php?q={cve_id}&api=True&rt=1"
        tm_response = requests.get(tm_url, timeout=10)
        if tm_response.status_code == 200:
            for result in tm_response.json().get('results', []):
                if result.get('type') == 'domain':
                    secondary_iocs['Domains'].add(result['value'])
                elif result.get('type') == 'ip':
                    secondary_iocs['IPs'].add(result['value'])
    
    except Exception as e:
        print(f"Error with ThreatMiner for {cve_id}: {str(e)}")
    
    return {k: list(v) for k, v in secondary_iocs.items() if v}

In [ ]:
get_iocsANDexploits('CVE-2017-5638')

In [ ]:
# MAIN

In [ ]:
# Find ways to integrate when it come from otx or data miner and find a way to avoid too many requiest to the API issue

# CVE to IOCs and exploits


# print(os.getcwd())
CVE_CWE_mappingDF = pd.read_csv('../CVE_CWE_mapping.csv', sep=',')

CVE_list = list(set(CVE_CWE_mappingDF['Key'].tolist()))  # Removes duplicates but loses original order

CVE_to_Exploit_N_IOC_dict = dict()

for CVE in CVE_list:
    CVE_to_Exploit_N_IOC_dict[CVE] = get_iocsANDexploits(CVE)

    
    

In [ ]:
# TTPs to CWEs

TTP_CWE_mappingDF = pd.read_csv('../658.csv', sep=',')

columns_list = TTP_CWE_mappingDF.columns.tolist()  # Returns column names as a list
print(columns_list)
print("\n")
pd.show_versions()

TEST = TTP_CWE_mappingDF.iloc[:, [0]]



columns_list2 = TEST.columns.tolist()  # Returns column names as a list
print(columns_list2)

print(TEST)

# first_column = TTP_CWE_mappingDF.iloc[:, 0]  # All rows (:) + first column (0)
# print(first_column)



In [43]:
# CWEs to CVEs

CVE_CWE_mapping_DF = pd.read_csv('../CVE_CWE_mapping.csv', sep=',')

# columns_list = CVE_CWE_mapping_DF.columns.tolist()  # Returns column names as a list
# print(columns_list)

# print(CVE_CWE_mapping_DF.head())


import pandas as pd
import ast  # For safely evaluating string literals

def expand_cwe_entries(df):
    """
    Expands rows where Value contains multiple CWEs into separate rows.
    Handles both string representations of lists and actual lists.
    """
    expanded_rows = []
    
    for _, row in df.iterrows():
        key = row['Key']
        values = row['Value']
        
        # Convert string representation of list to actual list if needed
        if isinstance(values, str):
            try:
                values = ast.literal_eval(values)
            except (ValueError, SyntaxError):
                values = [values]  # Treat as single-item list if not parsable
        
        # Ensure values is a list (in case it's a single string)
        if not isinstance(values, list):
            values = [values]
            
        # Create new row for each CWE
        for value in values:
            # Clean up value by removing any list brackets if they exist
            clean_value = str(value).replace('[', '').replace(']', '').strip()
            expanded_rows.append({'Key': key, 'Value': f"[{clean_value}]"})
    
    return pd.DataFrame(expanded_rows)

# Example usage:
# Assuming your original DataFrame is called 'vuln_df'
CVE_CWE_mapping_DF = expand_cwe_entries(CVE_CWE_mapping_DF)




# print(CVE_CWE_mapping_DF.head(10))

# Duplicate 'Key' column and name the copy 'Key_Copy'
CVE_CWE_mapping_DF['Value_Copy'] = CVE_CWE_mapping_DF['Value']
# print(CVE_CWE_mapping_DF.head(10))


for idx, element in enumerate(CVE_CWE_mapping_DF['Value_Copy']):
    matches = re.findall(r'CWE-\d+', str(element))
    
    if len(matches) == 0:
        CVE_CWE_mapping_DF.at[idx, 'Value_Copy'] = []
    elif len(matches) == 1:
        CVE_CWE_mapping_DF.at[idx, 'Value_Copy'] = matches[0]  # Store as single value
    else:
        # print(f"Row {idx} has multiple CWEs: {matches}")
        CVE_CWE_mapping_DF.at[idx, 'Value_Copy'] = matches  # Store as list
        # Optional: break if you want to stop at first multiple match
        # break
    

# print(CVE_CWE_mapping_DF.head(10))
CVE_CWE_mapping_DF = CVE_CWE_mapping_DF.drop('Value', axis=1)
CVE_CWE_mapping_DF = CVE_CWE_mapping_DF.rename(columns={'Value_Copy': 'Value'})

CVE_CWE_mapping_DF = expand_cwe_entries(CVE_CWE_mapping_DF)




for idx, element in enumerate(CVE_CWE_mapping_DF['Value']):
    element = element.replace('[', '')
    element = element.replace(']', '')
    CVE_CWE_mapping_DF.at[idx, 'Value'] = element
    


# CVE_CWE_mapping_DF.to_csv('../CVE_to_CWE.csv', index=False)  # Excludes row numbers



In [64]:
import pandas as pd
import re

# Read the CSV
CVE_CWE_mapping_DF = pd.read_csv('../CVE_to_CWE.csv', sep=',')

# Initialize as dictionary instead of list
CWE_to_CVE_dict = {}

# Get unique CWEs
CWE_list = CVE_CWE_mapping_DF['Value'].unique().tolist()

for CWE in CWE_list:
    # Use word boundaries for exact match
    pattern = rf'\b{re.escape(CWE)}\b'  # re.escape handles special regex chars
    matches = CVE_CWE_mapping_DF[CVE_CWE_mapping_DF['Value'].str.contains(
        pattern, regex=True, na=False)]
    
    # Store as list of CVEs
    CWE_to_CVE_dict[CWE] = matches['Key'].tolist()

# # Example: View first 5 entries
# for i, (cwe, cves) in enumerate(CWE_to_CVE_dict.items()):
#     if i >= 5:  # Limit output
#         break
#     print(f"{cwe}: {len(cves)} CVEs")

# print(CWE_to_CVE_dict)

# for key in CWE_to_CVE_dict:
#     print(CWE_to_CVE_dict[key])

In [71]:
# for key in CWE_to_CVE_dict:
#     print(key, ": ", CWE_to_CVE_dict[key], "\n")
#     print("\n")



with open('../CWE_to_CVE_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(CWE_to_CVE_dict, f, 
             indent=2, 
             ensure_ascii=False)